In [24]:
import pandas as pd
""
df = pd.read_csv('forest_reserve_state.csv')

df.head()

,date,state,area
0,2003-01-01,Johor,356922.0
1,2003-01-01,Kedah,344530.0
2,2003-01-01,Kelantan,629687.0
3,2003-01-01,Melaka,5468.0
4,2003-01-01,Negeri Sembilan,165639.0


In [25]:
# Convert date column to datetime & extract year
df['year'] = pd.to_datetime(df['date']).dt.year

# 🔹 Clean data first
df = df[df['state'] != 'Semenanjung Malaysia']

rename_map = {
    'W.P. Kuala Lumpur': 'Kuala Lumpur',
    'W.P. Labuan': 'Labuan',
    'W.P. Putrajaya': 'Putrajaya'
}
df['state'] = df['state'].replace(rename_map)

# (Optional) Reset index
df = df.reset_index(drop=True)

# 🔹 Now find latest year
latest_year = df['year'].max()

# 🔹 Filter for the latest year
df_latest = df[df['year'] == latest_year]

print("Latest year:", latest_year)
print(df_latest.head())

Latest year: 2021
           date            state      area  year
288  2021-01-01            Johor  334502.0  2021
289  2021-01-01            Kedah  341976.0  2021
290  2021-01-01         Kelantan  629881.0  2021
291  2021-01-01           Melaka    5199.0  2021
292  2021-01-01  Negeri Sembilan  155143.0  2021


In [26]:
df_latest.to_csv("forest_reserve_cleaned.csv", index=False)

In [1]:
import pandas as pd 

df = pd.read_csv("air_pollution.csv", parse_dates=["date"])

print(df.head())

        date pollutant  concentration
0 2017-01-01        CO          0.561
1 2017-02-01        CO          0.530
2 2017-03-01        CO          0.589
3 2017-04-01        CO          0.662
4 2017-05-01        CO            NaN


In [2]:
# Extract year and month
df['year'] = df['date'].dt.year

# make sure concentration is numeric (empty strings -> NaN)
df['concentration'] = pd.to_numeric(df['concentration'], errors='coerce')

df = df[df['year'].isin([2019, 2020, 2021])]

# drop missing concentration values 
df = df.dropna(subset=['concentration'])

# group by pollutant + year, compute average
agg = df.groupby(["pollutant", "year"], as_index=False)["concentration"].mean()

# build hierarchical structure
rows = []
rows.append({'id': 'All', 'parent': '', 'value': ''})   # root -> parent empty cell

for pollutant in sorted(agg['pollutant'].unique()):
    rows.append({'id': pollutant, 'parent': 'All', 'value': ''})
    subset = agg[agg['pollutant'] == pollutant].sort_values('year')
    for _, r in subset.iterrows():
        rows.append({
            'id': f"{pollutant}-{int(r['year'])}",
            'parent': pollutant,
            'value': round(float(r['concentration']), 6)
        })

out = pd.DataFrame(rows, columns=['id','parent','value'])

In [3]:
out.head(30)

,id,parent,value
0,All,,
1,CO,All,
2,CO-2019,CO,0.650083
3,CO-2020,CO,0.544917
4,CO-2021,CO,0.53375
5,NO2,All,
6,NO2-2019,NO2,0.007192
7,NO2-2020,NO2,0.005767
8,NO2-2021,NO2,0.005692
9,O3,All,


In [4]:
out.to_csv("pollutants_cleaned.csv", index=False)
print("Cleaned file saved as pollutants_cleaned.csv")

Cleaned file saved as pollutants_cleaned.csv


In [2]:
import pandas as pd

# Load rainfall dataset
rainfall = pd.read_csv("mean_temp_rainfall.csv")

# Load pollution dataset
pollution = pd.read_csv("air_pollution.csv")

print(rainfall.head())
print(pollution.head())

      State Selected meteorological station   \
0     Johor                            Senai   
1     Johor                           Kluang   
2     Kedah                       Alor Setar   
3     Kedah                   Pulau Langkawi   
4  Kelantan                       Kota Bharu   

  Height above mean sea level in metres  Year  \
0                               (37.8m)  2000   
1                               (88.1m)  2000   
2                                (3.9m)  2000   
3                                (6.4m)  2000   
4                                (4.4m)  2000   

  Minimum Mean temperature in Celcius Maximum Mean temperature in Celcius  \
0                                22.9                                32.3   
1                                23.1                                32.0   
2                                23.6                                32.5   
3                                25.0                                32.0   
4                              

In [4]:
### rainfall data
# Force rainfall to numeric (remove text, commas, etc. if any)
rainfall["Total Rainfall in millimetres"] = pd.to_numeric(
    rainfall["Total Rainfall in millimetres"], errors="coerce"
)

# Keep relevant columns
rainfall_clean = rainfall[["State", "Year", "Total Rainfall in millimetres"]]

# Average rainfall per state per year (since some states have multiple stations)
rainfall_yearly = rainfall_clean.groupby(["State", "Year"], as_index=False).mean(numeric_only=True)

rainfall_yearly.rename(columns={"Total Rainfall in millimetres": "Rainfall_mm"}, inplace=True)

### pollution data
# Extract year from date
pollution["Year"] = pd.to_datetime(pollution["date"]).dt.year

# Remove 2017
pollution = pollution[pollution["Year"] != 2017]

# Average pollution per year
pollution_yearly = pollution.groupby("Year", as_index=False)["concentration"].mean()

# Merge by year
merged = pd.merge(rainfall_yearly, pollution_yearly, on="Year", how="inner")

merged.head(20)

,State,Year,Rainfall_mm,concentration
0,Johor,2018,2266.525000,6.974543
1,Johor,2019,2174.525000,8.387444
2,Johor,2020,2458.800000,5.431035
3,Johor,2021,2635.200000,5.668204
4,Kedah,2018,2202.200000,6.974543
5,Kedah,2019,2296.950000,8.387444
6,Kedah,2020,2570.000000,5.431035
7,Kedah,2021,2272.000000,5.668204
8,Kelantan,2018,2592.100000,6.974543
9,Kelantan,2019,1758.066667,8.387444


In [5]:
# Create categories based on tertiles
merged["PollutionCategory"] = pd.qcut(merged["concentration"], q=3, labels=["Low", "Medium", "High"])

merged.head(20)

,State,Year,Rainfall_mm,concentration,PollutionCategory
0,Johor,2018,2266.525000,6.974543,Medium
1,Johor,2019,2174.525000,8.387444,High
2,Johor,2020,2458.800000,5.431035,Low
3,Johor,2021,2635.200000,5.668204,Low
4,Kedah,2018,2202.200000,6.974543,Medium
5,Kedah,2019,2296.950000,8.387444,High
6,Kedah,2020,2570.000000,5.431035,Low
7,Kedah,2021,2272.000000,5.668204,Low
8,Kelantan,2018,2592.100000,6.974543,Medium
9,Kelantan,2019,1758.066667,8.387444,High


In [6]:
merged.to_csv("rainfall_pollution_ready.csv", index=False)

In [14]:
import pandas as pd 

pollution = pd.read_csv("air_pollution_station.csv")
rainfall = pd.read_csv("mean_temp_rainfall.csv")

# --- Clean column names ---
pollution.columns = pollution.columns.str.strip()
rainfall.columns = rainfall.columns.str.strip()

# --- Convert numeric columns (this part comes BEFORE any groupby or merge) ---
pollution["Maximum"] = pd.to_numeric(pollution["Maximum"], errors="coerce")
pollution["Minimum"] = pd.to_numeric(pollution["Minimum"], errors="coerce")

rainfall["Total Rainfall in millimetres"] = pd.to_numeric(rainfall["Total Rainfall in millimetres"], errors="coerce")
rainfall["Number of Days of Rainfall"] = pd.to_numeric(rainfall["Number of Days of Rainfall"], errors="coerce")
rainfall["Mean relative humidity in Percentage"] = pd.to_numeric(rainfall["Mean relative humidity in Percentage"], errors="coerce")
rainfall["Maximum Mean temperature in Celcius"] = pd.to_numeric(rainfall["Maximum Mean temperature in Celcius"], errors="coerce")
rainfall["Minimum Mean temperature in Celcius"] = pd.to_numeric(rainfall["Minimum Mean temperature in Celcius"], errors="coerce")

# --- Compute average pollution per record ---
pollution["AvgPollution"] = (pollution["Maximum"] + pollution["Minimum"]) / 2

# --- If you have multiple stations per state, average them ---
pollution_state = pollution.groupby(["Year", "State"], as_index=False)["AvgPollution"].mean()

# --- Average rainfall per state per year ---
rainfall_summary = rainfall.groupby(["Year", "State"], as_index=False).agg({
    "Total Rainfall in millimetres": "mean",
    "Number of Days of Rainfall": "mean"
})
rainfall_summary.rename(columns={
    "Total Rainfall in millimetres": "Rainfall_mm",
    "Number of Days of Rainfall": "RainDays"
}, inplace=True)

# --- Merge pollution + rainfall ---
merged = pd.merge(pollution_state, rainfall_summary, on=["Year", "State"], how="inner")

# --- Export to CSV for Vega-Lite ---
merged.to_csv("pollution_vs_rainfall.csv", index=False)
print(merged.head(10))


KeyError: 'State'

In [1]:
import pandas as pd

# Load your raw data (replace filename with your actual path)
df = pd.read_csv("tree_cover_loss_by_driver.csv")

# Show a few rows
df.head()

,drivers_type,loss_year,loss_area_ha,gross_carbon_emissions_Mg
0,Hard commodities,2001,2242.427682,1.210344e+06
1,Logging,2001,49884.599047,3.083325e+07
2,Other natural disturbances,2001,368.290169,2.240721e+05
3,Permanent agriculture,2001,267004.924729,9.340107e+07
4,Settlements & Infrastructure,2001,5756.636416,2.563977e+06


In [5]:
df = df.rename(columns={
    "drivers_type": "driver_type",
    "loss_year": "year",
    "loss_area_ha": "loss_area_ha",
    "gross_carbon_emissions_Mg": "carbon_emission_Mg"
})

minor_types = ["Hard commodities", "Other natural disturbances", "Unknown"]

df["driver_type"] = df["driver_type"].replace(minor_types, "Other")

df = df.groupby(["driver_type", "year"], as_index=False).agg({
    "loss_area_ha": "sum",
    "carbon_emission_Mg": "sum"
})

main_drivers = [
    "Permanent agriculture",
    "Logging",
    "Shifting cultivation",
    "Settlements & Infrastructure",
    "Wildfire",
    "Other"
]

# Keep only the main ones
df = df[df["driver_type"].isin(main_drivers)]

df["year"] = df["year"].astype(int)
df["loss_area_ha"] = df["loss_area_ha"].astype(float)
df["carbon_emission_Mg"] = df["carbon_emission_Mg"].astype(float)

df.info()
df.to_csv("tree_loss_clean.csv", index=False)
print("Clean dataset saved as tree_loss_clean.csv")
df.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   driver_type         144 non-null    object 
 1   year                144 non-null    int64  
 2   loss_area_ha        144 non-null    float64
 3   carbon_emission_Mg  144 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 4.6+ KB
Clean dataset saved as tree_loss_clean.csv


,driver_type,year,loss_area_ha,carbon_emission_Mg
0,Logging,2001,49884.599047,3.083325e+07
1,Logging,2002,36866.309788,2.442155e+07
2,Logging,2003,34998.463405,2.545375e+07
3,Logging,2004,66820.315935,5.130144e+07
4,Logging,2005,54901.248742,4.300234e+07
5,Logging,2006,57025.864602,4.335688e+07
6,Logging,2007,80691.037935,6.265439e+07
7,Logging,2008,73625.258540,5.827702e+07
8,Logging,2009,97086.859639,7.642175e+07
9,Logging,2010,59684.214713,4.657186e+07


In [1]:
import pandas as pd 

# === 1️⃣ Load datasets ===
forest_raw = pd.read_csv("forest_area_annually.csv", skiprows=4)
pop_raw = pd.read_csv("population_growth.csv", skiprows=4)

print(forest_raw.head())
print(pop_raw.head())

                  Country Name Country Code                Indicator Name  \
0                        Aruba          ABW  Forest area (% of land area)   
1  Africa Eastern and Southern          AFE  Forest area (% of land area)   
2                  Afghanistan          AFG  Forest area (% of land area)   
3   Africa Western and Central          AFW  Forest area (% of land area)   
4                       Angola          AGO  Forest area (% of land area)   

   Indicator Code  1960  1961  1962  1963  1964  1965  ...       2016  \
0  AG.LND.FRST.ZS   NaN   NaN   NaN   NaN   NaN   NaN  ...   2.333333   
1  AG.LND.FRST.ZS   NaN   NaN   NaN   NaN   NaN   NaN  ...  31.039682   
2  AG.LND.FRST.ZS   NaN   NaN   NaN   NaN   NaN   NaN  ...   1.852782   
3  AG.LND.FRST.ZS   NaN   NaN   NaN   NaN   NaN   NaN  ...  20.152610   
4  AG.LND.FRST.ZS   NaN   NaN   NaN   NaN   NaN   NaN  ...  55.207845   

        2017       2018       2019       2020       2021       2022  \
0   2.333333   2.333333   2

In [3]:

# === Filter only Malaysia ===
forest_my = forest_raw[forest_raw["Country Name"] == "Malaysia"]
pop_my = pop_raw[pop_raw["Country Name"] == "Malaysia"]

# === Convert from wide → long format ===
forest_long = forest_my.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="Forest_area_percent"
)

pop_long = pop_my.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="Population_total"
)

# === Clean and convert year ===
forest_long["Year"] = pd.to_numeric(forest_long["Year"], errors="coerce")
pop_long["Year"] = pd.to_numeric(pop_long["Year"], errors="coerce")

# === Filter valid years (1990–2023) ===
forest_long = forest_long[(forest_long["Year"] >= 2001) & (forest_long["Year"] <= 2023)]
pop_long = pop_long[(pop_long["Year"] >= 2001) & (pop_long["Year"] <= 2023)]

# === Keep only useful columns ===
forest_clean = forest_long[["Year", "Forest_area_percent"]].dropna()
pop_clean = pop_long[["Year", "Population_total"]].dropna()

# === Merge both datasets ===
merged = pd.merge(forest_clean, pop_clean, on="Year", how="inner")

# === save cleaned CSVs ===
forest_clean.to_csv("cleaned_forest_area_malaysia.csv", index=False)
pop_clean.to_csv("cleaned_population_malaysia.csv", index=False)
merged.to_csv("merged_forest_population_malaysia.csv", index=False)

print("All cleaned datasets saved:")
print(" - cleaned_forest_area_malaysia.csv")
print(" - cleaned_population_malaysia.csv")
print(" - merged_forest_population_malaysia.csv")

All cleaned datasets saved:
 - cleaned_forest_area_malaysia.csv
 - cleaned_population_malaysia.csv
 - merged_forest_population_malaysia.csv


In [15]:
import pandas as pd

# Load data
forest = pd.read_csv("forest_reserve_state.csv")
air = pd.read_csv("air_pollution_station.csv")

print(forest.head())
print(air.head())


         date            state      area
0  2003-01-01            Johor  356922.0
1  2003-01-01            Kedah  344530.0
2  2003-01-01         Kelantan  629687.0
3  2003-01-01           Melaka    5468.0
4  2003-01-01  Negeri Sembilan  165639.0
   Year         Selected Stations  Maximum  Minimum
0  1998  Bandaraya Melaka, Melaka       92        4
1  1998      Cheras, Kuala Lumpur      140       10
2  1998     Kota Kinabalu, Sabah       459        1
3  1998         Kuching, Sarawak        90        3
4  1998       Larkin, Johor Bahru      116        7


In [16]:

# Convert date to year
forest['Year'] = pd.to_datetime(forest['date']).dt.year

# Keep relevant columns
forest = forest[['Year', 'state', 'area']]

# Standardize state names
forest['state'] = forest['state'].str.strip().str.title()

# Drop national-level rows
forest = forest[~forest['state'].isin(['Semenanjung Malaysia'])]

# Manually map stations to standard state names
station_to_state = {
    "Bandaraya Melaka, Melaka": "Melaka",
    "Cheras, Kuala Lumpur": "W.P. Kuala Lumpur",
    "Kota Kinabalu, Sabah": "Sabah",
    "Kuching, Sarawak": "Sarawak",
    "Larkin, Johor Bahru": "Johor",
    "Miri, Sarawak": "Sarawak",
    "Seberang Jaya, Pulau Pinang": "Pulau Pinang"
    # Add more if needed
}

# Strip extra spaces and map
air['station'] = air['Selected Stations'].str.strip()
air['state'] = air['station'].map(station_to_state)

# Keep relevant columns
air = air[['Year', 'state', 'Maximum', 'Minimum']]

# Compute mean API
air['mean_api'] = air[['Maximum', 'Minimum']].mean(axis=1)

# Aggregate by state-year
air = air.groupby(['Year', 'state'], as_index=False)['mean_api'].mean()

merged = pd.merge(forest, air, on=['Year', 'state'], how='inner')

# Categorize into 3 levels per year
merged['area_cat'] = merged.groupby('Year')['area'].transform(
    lambda x: pd.qcut(x, q=3, labels=['Low', 'Medium', 'High'])
)

# Optionally, save the range info for legend
legend_info = merged.groupby('Year')['area'].quantile([0, 0.33, 0.66, 1]).unstack()
legend_info.to_csv("forest_area_category_ranges.csv")

agg = merged.groupby(['Year', 'area_cat'], as_index=False)['mean_api'].mean()
agg.to_csv("merged_forest_api_category.csv", index=False)

print(legend_info.head())
print(agg.head())

      0.00     0.33       0.66       1.00
Year                                     
2003  61.0  5352.85  1318801.2  4836800.0
2004  61.0  5346.85  1311996.2  4806300.0
2005  61.0  5346.85  1320060.6  4779900.0
2006  61.0  5337.75  1325005.1  4778300.0
2007  61.0  5741.35  1355509.4  4692900.0
   Year area_cat  mean_api
0  2003      Low     67.75
1  2003   Medium     58.50
2  2003     High     42.00
3  2004      Low     70.25
4  2004   Medium     65.75


C:\Users\User\AppData\Local\Temp\ipykernel_35316\782043251.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg = merged.groupby(['Year', 'area_cat'], as_index=False)['mean_api'].mean()


In [8]:
import pandas as pd

# --- Step 1: Load datasets ---
forest = pd.read_csv('forest_reserve_state.csv')       # Forest reserve data
rainfall = pd.read_csv('mean_temp_rainfall.csv')       # Rainfall data

# --- Step 2: Prepare forest dataset ---
forest['Year'] = pd.to_datetime(forest['date']).dt.year
forest = forest[['Year', 'state', 'area']]
forest['state'] = forest['state'].str.strip().str.title()

# Optional: drop national-level rows
forest = forest[~forest['state'].isin([
    'Semenanjung Malaysia', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya'
])]

# --- Step 3: Prepare rainfall dataset ---
# Manual mapping of rainfall states/stations to forest states
state_mapping = {
    'Johor': 'Johor',
    'Kedah': 'Kedah',
    'Kelantan': 'Kelantan',
    'Melaka': 'Melaka',
    'Negeri Sembilan': 'Negeri Sembilan',
    'Pahang': 'Pahang',
    'Perak': 'Perak',
    'Perlis': 'Perlis',
    'Pulau Pinang': 'Pulau Pinang',
    'Sabah': 'Sabah',
    'Sarawak': 'Sarawak',
    'Selangor': 'Selangor',
    'Terengganu': 'Terengganu',
    'Wilayah Persekutuan Labuan': 'W.P. Labuan',
    'W.P. Kuala Lumpur': 'W.P. Kuala Lumpur'
}

# Map the states
rainfall['state'] = rainfall['State'].map(state_mapping)

# Drop rows where mapping failed
rainfall = rainfall.dropna(subset=['state'])

# Keep only necessary columns
rainfall = rainfall[['Year', 'state', 'Total Rainfall in millimetres']]

# --- Step 3a: Convert rainfall to numeric ---
rainfall['Total Rainfall in millimetres'] = pd.to_numeric(
    rainfall['Total Rainfall in millimetres'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
)

# Drop rows where conversion failed
rainfall = rainfall.dropna(subset=['Total Rainfall in millimetres'])

# --- Step 3b: Aggregate if multiple stations per state-year ---
rainfall = rainfall.groupby(['Year', 'state'], as_index=False)['Total Rainfall in millimetres'].mean()

# --- Step 4: Merge datasets ---
merged = pd.merge(forest, rainfall, on=['Year', 'state'], how='inner')

# --- Step 6: Save merged dataset ---
merged.to_csv('forest_rainfall_scatter.csv', index=False)

print("Merged dataset ready for scatter plot:")
print(merged.head(20))


Merged dataset ready for scatter plot:
    Year         state       area  Total Rainfall in millimetres
0   2003         Johor   356922.0                       2203.100
1   2003         Kedah   344530.0                       2704.250
2   2003      Kelantan   629687.0                       2534.850
3   2003        Melaka     5468.0                       1824.200
4   2003        Pahang  1524132.0                       3387.700
5   2003         Perak   884205.0                       2764.350
6   2003        Perlis    10718.0                       1837.300
7   2003  Pulau Pinang     5139.0                       2762.450
8   2003         Sabah  3563186.0                       2404.225
9   2003       Sarawak  4836800.0                       3950.750
10  2003      Selangor   233781.0                       2750.100
11  2003    Terengganu   535929.0                       3341.700
12  2004         Johor   355772.0                       2496.200
13  2004         Kedah   342613.0                  

In [11]:
import pandas as pd 

df = pd.read_csv('forest_reserve_state.csv')

# Convert date column to datetime & extract year
df['year'] = pd.to_datetime(df['date']).dt.year

# 🔹 Clean data first
df = df[df['state'] != 'Semenanjung Malaysia']

rename_map = {
    'W.P. Kuala Lumpur': 'Kuala Lumpur',
    'W.P. Labuan': 'Labuan',
    'W.P. Putrajaya': 'Putrajaya'
}
df['state'] = df['state'].replace(rename_map)

# (Optional) Reset index
df = df.reset_index(drop=True)

print(df.head())
df.to_csv('forest_reserve_cleaned.csv')

         date            state      area  year
0  2003-01-01            Johor  356922.0  2003
1  2003-01-01            Kedah  344530.0  2003
2  2003-01-01         Kelantan  629687.0  2003
3  2003-01-01           Melaka    5468.0  2003
4  2003-01-01  Negeri Sembilan  165639.0  2003


In [12]:
import pandas as pd

# Load your cleaned data
df = pd.read_csv("forest_reserve_cleaned.csv")

# See which years have missing values
missing_summary = df.pivot_table(
    index="state", columns="year", values="area", aggfunc="first"
)

# Count how many states are missing data per year
missing_counts = missing_summary.isna().sum()

print("Number of missing values per year:")
print(missing_counts)

# (Optional) Show which specific states are missing values
for year in missing_summary.columns:
    missing_states = missing_summary[missing_summary[year].isna()].index.tolist()
    if missing_states:
        print(f"\n{year} missing for: {missing_states}")


Number of missing values per year:
year
2003    0
2004    0
2005    0
2006    0
2007    0
2008    0
2009    0
2010    1
2011    1
2012    0
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
dtype: int64

2010 missing for: ['Sarawak']

2011 missing for: ['Sarawak']


In [7]:
import pandas as pd

df = pd.read_csv("forest_reserve_cleaned.csv")
df_wide = df.pivot(index="state", columns="year", values="area").reset_index()
df_wide.to_csv("forest_reserve_wide.csv", index=False)


In [13]:
import pandas as pd

# load your files (adjust paths if needed)
df = pd.read_csv("forest_reserve_cleaned.csv")        # long: state,year,area (ha)
state_area = pd.read_csv("malaysia_state_area.csv")  # should have columns: state,total_area_km2

# ensure year is integer
df["year"] = df["year"].astype(int)

# pivot area to wide
area_wide = df.pivot(index="state", columns="year", values="area").reset_index()

# merge total area (km2) to compute percent
area_wide = area_wide.merge(state_area, on="state", how="left")

# compute pct columns and optionally keep area columns
years = [2003, 2009, 2015, 2021]   # use whichever years you want
for y in years:
    if y in area_wide.columns:
        area_col = y
        pct_col = f"{y}_pct"
        # area is in hectares in your data — convert to fraction of state area:
        # state total_area_km2 * 100 -> hectares (1 km2 = 100 ha)
        area_wide[pct_col] = area_wide[area_col] / (area_wide["total_area_km2"] * 100)
    else:
        area_wide[f"{y}_pct"] = pd.NA
        print(f"Warning: year {y} not present in data")

# optionally drop the raw area columns or keep them
# save CSV for Vega-Lite lookup
area_wide.to_csv("forest_reserve_wide_pct.csv", index=False)
print("Saved forest_reserve_wide_pct.csv with columns:", list(area_wide.columns))


Saved forest_reserve_wide_pct.csv with columns: ['state', 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 'total_area_km2', '2003_pct', '2009_pct', '2015_pct', '2021_pct']


In [15]:
import pandas as pd
import numpy as np

# load your wide CSV (must contain columns like '2003_pct','2009_pct','2015_pct','2021_pct')
df = pd.read_csv("forest_reserve_wide_pct.csv")  # replace path if different

# years to convert (pct columns must exist)
years = [2003, 2009, 2015, 2021]
pct_cols = [f"{y}_pct" for y in years]
class_cols = [f"{y}_class" for y in years]

# define classification function
def pct_to_class(p):
    # handle missing or NaN
    if pd.isna(p):
        return ""      # empty string will render as no fill / missing
    try:
        p = float(p)
    except Exception:
        return ""
    if p <= 0.05:
        return "Very Low (0–5%)"
    if p <= 0.20:
        return "Low (5–20%)"
    if p <= 0.35:
        return "Medium (20–35%)"
    if p <= 0.45:
        return "High (35–45%)"
    return "Very High (45%+)"

# compute class columns
for pct_col, class_col in zip(pct_cols, class_cols):
    if pct_col not in df.columns:
        print(f"Warning: {pct_col} not found in dataframe. Creating empty {class_col}.")
        df[class_col] = ""
    else:
        df[class_col] = df[pct_col].apply(pct_to_class)

# Optionally inspect rows with missing pct
missing = {}
for pct_col in pct_cols:
    if pct_col in df.columns:
        missing[pct_col] = df[pct_col].isna().sum()
print("Missing values per pct column:", missing)

# save ready CSV
df.to_csv("forest_reserve_ready.csv", index=False)
print("Saved forest_reserve_ready.csv with class columns:", class_cols)


Missing values per pct column: {'2003_pct': np.int64(0), '2009_pct': np.int64(0), '2015_pct': np.int64(0), '2021_pct': np.int64(0)}
Saved forest_reserve_ready.csv with class columns: ['2003_class', '2009_class', '2015_class', '2021_class']


In [2]:
import pandas as pd

# Read raw dataset
df = pd.read_csv("viirs_fire_alerts__count.csv")

# --- Step 1: Create a 'alert_date' column from year + week ---
# ISO weeks start on Monday; use `pd.to_datetime` with 'W' format
df["alert_date"] = pd.to_datetime(df["alert__year"].astype(str) + df["alert__week"].astype(str) + "-1", format="%G%V-%u")

# --- Step 2: (Optional) Filter out invalid or future years ---
df = df[(df["alert__year"] >= 2012) & (df["alert__year"] <= 2024)]

# --- Step 3: Keep only useful columns ---
df = df[["alert_date", "alert__year", "alert__week", "alert__count", "confidence__cat"]]

# --- Step 4: (Optional) Aggregate if duplicate weeks exist ---
df = df.groupby(["alert_date", "alert__year", "alert__week", "confidence__cat"], as_index=False)["alert__count"].sum()

# --- Step 5: Add derived columns for easier Vega-Lite filtering ---
df["month"] = df["alert_date"].dt.month
df["year_month"] = df["alert_date"].dt.to_period("M").astype(str)  # e.g., '2012-03'

# --- Step 6: Save cleaned dataset ---
df.to_csv("malaysia_fire_alerts_clean.csv", index=False)

print(df.head())


  alert_date  alert__year  alert__week confidence__cat  alert__count  month  \
0 2012-01-23         2012            4               h             1      1   
1 2012-01-30         2012            5               h             1      1   
2 2012-02-06         2012            6               h             2      2   
3 2012-02-13         2012            7               h             2      2   
4 2012-02-27         2012            9               h             1      2   

  year_month  
0    2012-01  
1    2012-01  
2    2012-02  
3    2012-02  
4    2012-02  


In [1]:
import pandas as pd

# Read raw dataset
df = pd.read_csv("malaysia_fire_alerts_clean.csv")

# --- Step 1: Create a 'alert_date' column from year + week ---
df["alert_date"] = pd.to_datetime(df["alert__year"].astype(str) + df["alert__week"].astype(str) + "-1", format="%G%V-%u")

# --- Step 2: (Optional) Filter out invalid or future years ---
df = df[(df["alert__year"] >= 2012) & (df["alert__year"] <= 2024)]

# --- Step 3: Keep only useful columns ---
df = df[["alert_date", "alert__year", "alert__week", "alert__count", "confidence__cat"]]

# --- Step 4: (Optional) Aggregate if duplicate weeks exist ---
df = df.groupby(["alert_date", "alert__year", "alert__week", "confidence__cat"], as_index=False)["alert__count"].sum()

# --- Step 5: Add derived columns for Vega-Lite filtering ---
df["month"] = df["alert_date"].dt.month
df["year_month"] = df["alert_date"].dt.to_period("M").astype(str)
df["year_month_day"] = df["alert_date"].dt.strftime("%Y-%m-%d")   # 👈 add formatted date

# --- Step 6: Save cleaned dataset ---
df.to_csv("malaysia_fire_alerts_clean.csv", index=False)

print(df.head())


  alert_date  alert__year  alert__week confidence__cat  alert__count  month  \
0 2012-01-23         2012            4               h             1      1   
1 2012-01-30         2012            5               h             1      1   
2 2012-02-06         2012            6               h             2      2   
3 2012-02-13         2012            7               h             2      2   
4 2012-02-27         2012            9               h             1      2   

  year_month year_month_day  
0    2012-01     2012-01-23  
1    2012-01     2012-01-30  
2    2012-02     2012-02-06  
3    2012-02     2012-02-13  
4    2012-02     2012-02-27  
